## 1. Load the data

In [ ]:
# Load pandas, the standard library for working with tables (dataframes)
import pandas as pd

# Show full text in cells instead of cutting long names off with "..."
pd.set_option("display.max_colwidth", None)

# Read the main data file. "../" goes up from notebooks/ to the project root
df = pd.read_csv("../data/raw/24100051.csv")

In [ ]:
# Rows x columns, to get a sense of size
print(df.shape)

# All column names as a plain list
print(df.columns.tolist())

# First 10 rows. Left unwrapped (no print) so it renders as a formatted table
df.head(10)

In [ ]:
# Count how often each value appears. Confirms the unit (units vs thousands)
df["SCALAR_FACTOR"].value_counts()

In [ ]:
# Which geographies are in the data, and how many rows each has
df["GEO"].value_counts()

In [ ]:
# Every country/region member and its row count.
# Uneven counts hint that some members only exist at certain geographies
df["Country of residence"].value_counts()

In [ ]:
# Test the hunch: how many rows does Iran have at each geography?
# groupby splits the rows by GEO, .size() counts the rows in each group
df[df["Country of residence"] == "Iran"].groupby("GEO").size()

In [ ]:
# Keep only the rows where STATUS has a flag (i.e. is not empty)
flagged = df[df["STATUS"].notna()]

# Which months are affected? If it's a single month, it's a one-off gap
flagged["REF_DATE"].unique()

In [ ]:
# Earliest and latest month. Works on text because the format is YYYY-MM,
# so alphabetical order matches time order
df["REF_DATE"].min(), df["REF_DATE"].max()

## 2. Parse the metadata hierarchy

In [23]:
import re
from io import StringIO
from pathlib import Path

# Read the whole metadata file as one long string of text.
# "utf-8-sig" quietly removes the hidden BOM character if one is present
text = Path("../data/raw/24100051_MetaData.csv").read_text(encoding="utf-8-sig")

# Split the text into sections wherever there's a blank line.
# The regex means: newline, any spaces, newline (handles "blank" lines with spaces)
sections = re.split(r"\n\s*\n", text)

# How many sections, and what does each one start with?
for i, s in enumerate(sections):
    print(i, s[:80])  # first 80 characters of each section

0 "Cube Title","Product Id","CANSIM Id",URL,"Cube Notes","Archive Status",Frequenc
1 "Dimension ID","Dimension name","Dimension Notes","Dimension Definitions"
"1","G
2 "Dimension ID","Member Name","Classification Code","Member ID","Parent Member ID
3 Symbol Legend
Description,Symbol
"not available for a specific reference period"
4 "Survey Code","Survey Name"
"5005","Frontier Counts"
5 "Subject Code","Subject Name"
"24","Travel and tourism"
6 "Note ID",Note
1,"Seasonally-adjusted data have been produced using X12 ARIMA se
7 "Correction ID","Correction Date","Correction Note"



In [24]:
# Find the one section whose header starts like the member table.
# next(...) returns the first match, and raises StopIteration if nothing matches,
# which is the loud failure we want if StatCan ever changes the format
members_text = next(
    s for s in sections if s.startswith('"Dimension ID","Member Name"')
)

# StringIO makes a string behave like a file, so read_csv can parse it
# (same idea as BytesIO for the zip, but for text)
members = pd.read_csv(StringIO(members_text))

members.head(15)

,Dimension ID,Member Name,Classification Code,Member ID,Parent Member ID,Terminated,Member Notes,Member Definitions
0,1,Canada,[11124],1,NaN,NaN,NaN,NaN
1,1,Quebec,[24],2,1.0,NaN,NaN,NaN
2,1,Ontario,[35],3,1.0,NaN,NaN,NaN
3,1,British Columbia,[59],4,1.0,NaN,NaN,NaN
4,1,Other provinces and territories,NaN,5,1.0,NaN,14.0,NaN
5,2,Non-resident visitors entering Canada,NaN,1,NaN,NaN,NaN,NaN
6,2,United States of America residents entering Ca...,NaN,2,1.0,NaN,NaN,NaN
7,2,Residents of countries other than the United S...,NaN,3,1.0,NaN,NaN,NaN
8,2,"Americas, countries other than the United Stat...",NaN,4,3.0,NaN,NaN,NaN
9,2,"North America, countries other than the United...",NaN,5,4.0,NaN,NaN,NaN


In [25]:
# Keep only dimension 2 (Country of residence); dimension 1 is Geography.
# .copy() makes this an independent table, so adding columns later is clean
countries = members[members["Dimension ID"] == 2].copy()

# Every ID that appears as someone's parent.
# dropna() removes the blank parent of the top-level total;
# astype(int) makes them whole numbers so they compare cleanly with Member ID
parent_ids = set(countries["Parent Member ID"].dropna().astype(int))

# A leaf is a member that nobody lists as their parent.
# ~ means "not", so: True where Member ID is NOT in the parent set
countries["is_leaf"] = ~countries["Member ID"].isin(parent_ids)

countries[["Member ID", "Member Name", "Parent Member ID", "is_leaf"]]

,Member ID,Member Name,Parent Member ID,is_leaf
5,1,Non-resident visitors entering Canada,NaN,False
6,2,United States of America residents entering Ca...,1.0,True
7,3,Residents of countries other than the United S...,1.0,False
8,4,"Americas, countries other than the United Stat...",3.0,False
9,5,"North America, countries other than the United...",4.0,False
...,...,...,...,...
89,85,Pakistan,82.0,True
90,86,Oceania,3.0,False
91,87,Australasia,86.0,False
92,88,Australia,87.0,True


In [26]:
# Show only the leaves, every row (no "..." truncation)
leaves = countries[countries["is_leaf"]]

# How many leaves are there? True counts as 1, so sum() counts them
print(leaves["is_leaf"].sum())

# to_string() prints the whole table instead of a shortened preview
print(leaves[["Member ID", "Member Name", "Parent Member ID"]].to_string())

66
    Member ID                                         Member Name  Parent Member ID
6           2  United States of America residents entering Canada               1.0
10          6                                              Mexico               5.0
11          7                           Saint Pierre and Miquelon               5.0
12          8                                     Central America               4.0
14         10                                             Bahamas               9.0
15         11                                            Barbados               9.0
16         12                                             Bermuda               9.0
17         13                                               Haiti               9.0
18         14                                             Jamaica               9.0
19         15                                 Trinidad and Tobago               9.0
21         17                                           Argentina        

In [27]:
# Pick one month at the national level to test
month = df[(df["GEO"] == "Canada") & (df["REF_DATE"] == "2024-07")]

# Sum the VALUE of only the leaf members
leaf_sum = month[month["Country of residence"].isin(leaves["Member Name"])]["VALUE"].sum()

# Grab the published total for the same month.
# .iloc[0] pulls the single value out of the one-row result
total = month.loc[
    month["Country of residence"] == "Non-resident visitors entering Canada", "VALUE"
].iloc[0]

# Compare them: the difference should be zero... or should it?
print(f"Leaf sum: {leaf_sum:,.0f}")
print(f"Total:    {total:,.0f}")
print(f"Diff:     {leaf_sum - total:,.0f}")

Leaf sum: 2,435,054
Total:    2,463,115
Diff:     -28,061


In [28]:
# Rows in this month whose country is a leaf
matched = month[month["Country of residence"].isin(leaves["Member Name"])]

# Should be 66 if every leaf found its row
print(len(matched))

# Any leaf names that did NOT find a row this month?
# set difference: names in the leaf list minus names actually present
print(set(leaves["Member Name"]) - set(month["Country of residence"]))

66
set()


In [29]:
# Helper: look up one member's value for this month by its name
def val(name):
    return month.loc[month["Country of residence"] == name, "VALUE"].iloc[0]

us = val("United States of America residents entering Canada")
non_us = val("Residents of countries other than the United States of America entering Canada")

# If even this simple split doesn't add up, the gap comes from the adjustment,
# not from missing countries
print(f"US + non-US: {us + non_us:,.0f}")
print(f"Total:       {total:,.0f}")
print(f"Diff:        {us + non_us - total:,.0f}")

US + non-US: 2,463,114
Total:       2,463,115
Diff:        -1


In [30]:
# Lookup table: country name -> VALUE for this month
values = month.set_index("Country of residence")["VALUE"]

# Lookup: Member ID -> Member Name, so we can name each parent
id_to_name = dict(zip(countries["Member ID"], countries["Member Name"]))

rows = []
# groupby on Parent Member ID gives each parent with its group of direct children.
# The top-level total has no parent (NaN), so groupby skips it automatically
for parent_id, kids in countries.groupby("Parent Member ID"):
    parent_name = id_to_name[int(parent_id)]
    parent_value = values[parent_name]                 # what StatCan published for the parent
    children_sum = values[kids["Member Name"]].sum()   # what its children add up to
    rows.append({
        "parent": parent_name,
        "n_children": len(kids),
        "parent_value": parent_value,
        "children_sum": children_sum,
        "gap": parent_value - children_sum,            # positive = parent is bigger than its parts
    })

# One row per parent, biggest gaps first
gaps = pd.DataFrame(rows).sort_values("gap", ascending=False)
gaps


,parent,n_children,parent_value,children_sum,gap
4,Caribbean,6,17801.0,11899.0,5902.0
20,Southern Asia,3,48201.0,44634.0,3567.0
8,Central/Eastern Europe,6,12383.0,9274.0,3109.0
18,Northeast Asia,5,71896.0,69188.0,2708.0
17,Middle East,4,10017.0,7435.0,2582.0
13,Western Africa,1,6939.0,4465.0,2474.0
5,South America,6,25381.0,23292.0,2089.0
10,Southern Europe,4,22639.0,20656.0,1983.0
14,Northern Africa,2,8611.0,7484.0,1127.0
7,Western Europe,6,107225.0,106618.0,607.0


## 3. Design checks for transform

In [31]:
# How many distinct members have real values (not blank) at each geography?
# If Canada has far more than the provinces, the hierarchy depth differs by GEO
has_value = df[df["VALUE"].notna()]
has_value.groupby("GEO")["Country of residence"].nunique()

GEO
British Columbia                    8
Canada                             89
Ontario                             8
Other provinces and territories     8
Quebec                              8
Name: Country of residence, dtype: int64

In [32]:
# Which members does Quebec actually report?
has_value.loc[has_value["GEO"] == "Quebec", "Country of residence"].unique()

<StringArray>
[                                         'Non-resident visitors entering Canada',
                             'United States of America residents entering Canada',
 'Residents of countries other than the United States of America entering Canada',
                    'Americas, countries other than the United States of America',
                                                                         'Europe',
                                                                         'Africa',
                                                                           'Asia',
                                                                        'Oceania']
Length: 8, dtype: str

In [33]:
# What type did pandas give the COORDINATE column?
df["COORDINATE"].dtype

dtype('float64')

In [34]:
# dtype= overrides pandas' type guessing for specific columns.
# Keeping COORDINATE as text preserves "1.10" exactly
df = pd.read_csv("../data/raw/24100051.csv", dtype={"COORDINATE": str})

# Split "1.10" on the dot into two columns, then convert each to an integer
df[["geo_id", "member_id"]] = df["COORDINATE"].str.split(".", expand=True).astype(int)

# Sanity check: each member_id should map to exactly one country name
df.groupby("member_id")["Country of residence"].nunique().max()  # should be 1

np.int64(1)